# Kubeflow SDK: End-to-End ML Workflow Tutorial

This tutorial demonstrates how the various clients in the Kubeflow SDK work together to form a complete Machine Learning workflow:

1. **Hyperparameter Tuning (`OptimizerClient`)**: Search for the best training parameters (e.g., learning rate) using Katib.
2. **Final Model Training (`TrainerClient`)**: Train the final model using the best parameters found.
3. **Model Registration (`ModelRegistryClient`)**: Register the trained model in the Kubeflow Model Registry.

By the end of this notebook, you will understand how to orchestrate these steps using clean Python APIs.

## Step 1: Initialize Clients and Setup

We start by importing the necessary clients from the Kubeflow SDK. If running in a Kubernetes cluster, the clients automatically use the in-cluster configuration or your active kubeconfig.

In [ ]:
import os

from kubeflow.hub import ModelRegistryClient
from kubeflow.optimizer import OptimizerClient
from kubeflow.trainer import TrainerClient

# Initialize SDK clients
trainer_client = TrainerClient()
optimizer_client = OptimizerClient()

print("Clients initialized successfully!")

## Step 2: Hyperparameter Tuning

First, we define a training function that accepts hyperparameters (`lr` and `batch_size`) as arguments. This function will be executed by Katib across different trials with varying parameter assignments.

In [ ]:
def trial_train_fn(lr: float, batch_size: int):
    import time

    print(f"Starting trial training with learning_rate={lr}, batch_size={batch_size}")

    # Simulate a standard epoch-based training loop
    for epoch in range(1, 4):
        loss = 1.0 / (epoch * lr * batch_size)
        accuracy = 0.5 + (0.45 * epoch / 3)

        # Output formatted metrics for Katib metrics collector sidecar
        print(f"epoch={epoch}")
        print(f"loss={loss:.4f}")
        print(f"accuracy={accuracy:.4f}")
        time.sleep(1)

    print("Trial training completed successfully!")

Now we define the optimization job config. We specify the search space for `lr` and `batch_size`, our objective metric (`loss`), and execution limits.

In [ ]:
from kubeflow.optimizer import Direction, Objective, Search, TrialConfig
from kubeflow.trainer import CustomTrainer, TrainJobTemplate

# Create the trial template using CustomTrainer
trial_template = TrainJobTemplate(
    trainer=CustomTrainer(func=trial_train_fn, func_args={"lr": 0.01, "batch_size": 32}),
)

# Define hyperparameter search spaces
search_space = {
    "lr": Search.uniform(min=0.001, max=0.05),
    "batch_size": Search.choice([16, 32]),
}

# Configure objective and trial constraints
objectives = [Objective(metric="loss", direction=Direction.MINIMIZE)]
trial_config = TrialConfig(num_trials=2, parallel_trials=1)

# Submit the Optimization Job (with fallback if Katib is not deployed in test environment)
try:
    opt_job_name = optimizer_client.optimize(
        trial_template=trial_template,
        search_space=search_space,
        objectives=objectives,
        trial_config=trial_config,
    )
    print(f"Submitted OptimizationJob: {opt_job_name}")
    print("Waiting for optimization job to complete...")
    optimizer_client.wait_for_job_status(opt_job_name)
    print("Optimization job complete!")
    best_results = optimizer_client.get_best_results(opt_job_name)
except Exception as e:
    print(
        f"OptimizerClient/Katib not available or failed ({e}). Falling back to selected hyperparameter for demonstration."
    )
    best_results = None

We inspect the optimization results and extract the optimal hyperparameters.

In [ ]:
if best_results:
    print(f"Best parameters found: {best_results.parameters}")
    print(f"Best metrics achieved: {best_results.metrics}")
else:
    print("No optimal trial results found. Using default tuned parameters.")

## Step 3: Train the Final Model

With the best hyperparameters retrieved, we submit a final `TrainJob` using `TrainerClient` to train our production-ready model.

In [ ]:
# Extract best parameters with fallbacks
best_lr = (
    float(best_results.parameters["lr"])
    if best_results and "lr" in best_results.parameters
    else 0.01
)
best_batch_size = (
    int(best_results.parameters["batch_size"])
    if best_results and "batch_size" in best_results.parameters
    else 32
)

print(f"Training final model with best hyperparameters: lr={best_lr}, batch_size={best_batch_size}")


def final_train_fn(lr: float, batch_size: int):
    import os
    import time

    print(f"Starting final training with learning_rate={lr}, batch_size={batch_size}")

    # Simulate final training loop
    for epoch in range(1, 4):
        loss = 0.8 / (epoch * lr * batch_size)
        accuracy = 0.6 + (0.35 * epoch / 3)
        print(f"Epoch {epoch}: loss={loss:.4f}, accuracy={accuracy:.4f}")
        time.sleep(1)

    # In a real training function, you would save your model to a remote storage (S3, GCS, PVC)
    # For illustration, we simulate writing a model artifact file
    os.makedirs("/tmp/model", exist_ok=True)
    with open("/tmp/model/model.txt", "w") as f:
        f.write(f"Model trained with learning_rate={lr}, batch_size={batch_size}\n")
    print("Final model saved to /tmp/model/model.txt")


# Submit final TrainJob
final_job_name = trainer_client.train(
    trainer=CustomTrainer(
        func=final_train_fn, func_args={"lr": best_lr, "batch_size": best_batch_size}
    ),
)
print(f"Submitted final TrainJob: {final_job_name}")

print("Waiting for final training job to complete...")
trainer_client.wait_for_job_status(final_job_name)
print("Final training complete!")

## Step 4: Model Registration

After training is complete, we register the model version and its artifact URI in the Kubeflow Model Registry.

To ensure this notebook is robust and can run cleanly in testing environments where the Model Registry service may not be deployed, we will check registry connectivity and fallback to a mock client if it is unavailable.

In [ ]:
from unittest.mock import MagicMock

# Determine Model Registry host/port
mr_host = os.environ.get(
    "MODEL_REGISTRY_URL", "http://model-registry-service.kubeflow.svc.cluster.local:8080"
)

try:
    print(f"Connecting to Model Registry at {mr_host}...")
    mr_client = ModelRegistryClient(base_url=mr_host)
    # Test connectivity by listing models
    list(mr_client.list_models())
    is_mock = False
    print("Connected to Model Registry successfully!")
except Exception as e:
    print(f"Could not connect to Model Registry: {e}.")
    print("Falling back to mock ModelRegistryClient for demo/testing purposes.")
    is_mock = True

if is_mock:
    # Setup a mock registry client with matching interfaces
    class MockModelRegistryClient:
        def register_model(
            self,
            name,
            uri,
            version,
            model_format_name=None,
            model_format_version=None,
            version_description=None,
        ):
            print(f"[MOCK MR] Registering model '{name}' (version {version}) from URI '{uri}'")
            mock_model = MagicMock()
            mock_model.name = name
            mock_model.version = version
            mock_model.uri = uri
            mock_model.id = "mock-id-12345"
            return mock_model

    mr_client = MockModelRegistryClient()

# Define registry registration parameters
model_name = "mnist-classifier"
model_version = "v1.0.0"
model_uri = "s3://my-bucket/models/mnist-classifier"

# Register model version in registry
registered_model = mr_client.register_model(
    name=model_name,
    uri=model_uri,
    version=model_version,
    model_format_name="pytorch",
    model_format_version="2.0",
    version_description="MNIST PyTorch classifier trained with optimized learning rate",
)

print(f"Model '{model_name}' version '{model_version}' has been successfully registered!")